In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
from mrc.io import get_tomo, save_tomo
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.measure import label, regionprops
from scipy.ndimage import binary_erosion, binary_dilation, binary_closing
from skimage.morphology import ball
import json
from scipy.spatial.distance import cdist
import numpy as np

In [3]:
def find_peak(semantic_mask):
    distance_map = ndimage.distance_transform_edt(semantic_mask)
    local_maxi = peak_local_max(distance_map, min_distance=4, threshold_abs=0.2, exclude_border=False)
    return distance_map, local_maxi

In [4]:
semantic_path = f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT.mrc'
semantic_mask = get_tomo(semantic_path)
# distance_map, local_maxi = find_peak(semantic_mask)

In [5]:
distance_map, local_maxi = find_peak(semantic_mask)

In [ ]:
# save_tomo(distance_map, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/distance_map.mrc')
center_distance_map = np.zeros_like(semantic_mask, dtype=np.int8)
center_distance_map[distance_map > 4] = 1
instance_label = label(center_distance_map)
# save_tomo(instance_label, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/instance_label.mrc')
# instance_label = label(center_distance_map)

In [12]:
flat = instance_label.ravel()

# 1) 统计每个标签的像素数（比 unique + counts 更快）
counts = np.bincount(flat)
# 2) 挑出“像素数 >= 300” 的大标签
large_labels = np.nonzero(counts >= 300)[0]

# 3) 一次性保留大标签，其它全部置 0
#    用映射表：下标是原标签，值是要映射到的新标签（不是大标签就映射到 0）
mapping = np.zeros_like(counts, dtype=instance_label.dtype)
mapping[large_labels] = large_labels
# 4) 重映射一把，底层 C 实现，速度极快
flat[:] = mapping[flat]

new_instance_label = flat.reshape(instance_label.shape)

save_tomo(new_instance_label, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/new_instance_label.mrc')


In [6]:
# 创建一个3D球形结构元素，半径为1
structuring_element = ball(3)
# 执行腐蚀操作
eroded_mask = binary_erosion(semantic_mask, structure=structuring_element)
eroded_closing_mask = binary_closing(eroded_mask, structure=structuring_element)

isinstance_mask = label(eroded_closing_mask)
save_tomo(isinstance_mask, 
          '/media/liushuo/data1/data/actin trace/actintomo/20200820/tomo/pp1033/MT/PO/pp1033_instance_label.mrc')

In [7]:
def remove_small_objects(binary_mask, min_size=500):
    isinstance_mask = label(binary_mask)
    props = regionprops(isinstance_mask)

    # Create a new array to hold only large objects
    final_mask = np.zeros_like(binary_mask)

    for prop in props:
        if prop.area >= min_size:
            # If the region size is greater than or equal to min_size, keep it
            print(f"Keeping label {prop.label} with area {prop.area}.")
            final_mask[isinstance_mask == prop.label] = prop.label
            
    final_mask = final_mask > 0
    return final_mask

In [8]:
remove_small_objects_mask = remove_small_objects(eroded_closing_mask, min_size=1000)
final_instance_mask = label(remove_small_objects_mask)
save_tomo(final_instance_mask, '/media/liushuo/data1/data/actin trace/actintomo/20200820/tomo/pp1033/MT/PO/pp1033_final_instance_label.mrc')

Keeping label 2 with area 51244.0.
Keeping label 4 with area 8612.0.
Keeping label 5 with area 44080.0.
Keeping label 6 with area 60769.0.
Keeping label 7 with area 49797.0.
Keeping label 8 with area 50764.0.
Keeping label 9 with area 3740.0.
Keeping label 10 with area 56085.0.
Keeping label 11 with area 5047.0.
Keeping label 16 with area 5234.0.
Keeping label 18 with area 1473.0.


In [9]:
distance_map, local_maxi = find_peak(remove_small_objects_mask)
print(local_maxi)
print(distance_map.shape)

[[137  66 303]
 [137  67 308]
 [136  63 293]
 ...
 [132  84 327]
 [ 51 416  13]
 [124  52 226]]
(214, 928, 960)


In [20]:
save_tomo(distance_map, 
          '/media/liushuo/data1/data/actin trace/actintomo/20200820/tomo/pp1033/MT/pp1033_distance_map.mrc')
markers = np.zeros_like(semantic_mask, dtype=int)
for i, peak in enumerate(local_maxi):
    markers[tuple(peak)] = i + 1  # 标记每个局部最大值为唯一的实例ID
save_tomo(markers, 
          '/media/liushuo/data1/data/actin trace/actintomo/20200820/tomo/pp1033/MT/pp1033_keypoints.mrc')

In [15]:
def euclidean_distance_matrix(points):
    """
    计算三维空间中所有点之间的欧几里得距离，并返回一个距离矩阵
    """
    diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
    dist_matrix = np.linalg.norm(diff, axis=2)
    return dist_matrix

def tsp_3d(points):
    """
    基于贪心算法计算在三维空间中按最短距离连接点的顺序，返回排序后的points列表
    尝试每个点作为起点，找到最短的路径。
    """
    n = len(points)
    dist_matrix = euclidean_distance_matrix(points)
    
    # 初始化最小路径和排序
    min_total_length = float('inf')
    best_order = []

    # 尝试每个点作为起点
    for start_idx in range(n):
        visited = [False] * n
        order = []
        current_idx = start_idx
        visited[current_idx] = True
        order.append(points[current_idx])
        total_length = 0

        for _ in range(n - 1):
            # 当前点到其他未访问点的距离
            dist_to_current = dist_matrix[current_idx]
            
            # 选择最近的未访问点
            nearest_idx = np.argmin([dist_to_current[i] if not visited[i] else np.inf for i in range(n)])
            
            # 更新路径
            current_idx = nearest_idx
            visited[current_idx] = True
            order.append(points[current_idx])
            total_length += dist_to_current[current_idx]

        # 比较得到最小路径
        if total_length < min_total_length:
            min_total_length = total_length
            best_order = order

    return best_order

In [16]:
def find_instance_for_keypoints(instance_mask, keypoints_list):
    """
    根据instance_mask为每个关键点分配对应的实例ID
    """
    instance_ids = []
    for kp in keypoints_list:
        z, y, x = kp
        instance_id = instance_mask[int(z), int(y), int(x)]
        instance_ids.append(instance_id)
    return instance_ids

def sort_keypoints_by_distance(keypoints):
    """
    根据最短距离排序关键点，使点从头到尾连成的距离最短
    使用贪心算法，选择最近的点连接。
    """
    # 初始化第一个点
    sorted_keypoints = [keypoints[0]]
    keypoints_remaining = set(map(tuple, keypoints[1:]))  # 剩余的点
    current_point = keypoints[0]

    while keypoints_remaining:
        # 计算当前点到剩余点的距离
        distances = cdist([current_point], list(keypoints_remaining))
        # 找到最近的点
        next_point = list(keypoints_remaining)[np.argmin(distances)]
        sorted_keypoints.append(next_point)
        keypoints_remaining.remove(tuple(next_point))
        current_point = next_point
    
    return np.array(sorted_keypoints)

def calculate_total_length(sorted_keypoints):
    """
    计算按最短路径排序后的关键点总长度
    """
    length = 0
    for i in range(1, len(sorted_keypoints)):
        length += np.linalg.norm(sorted_keypoints[i] - sorted_keypoints[i - 1])
    return length

def save_results_to_json(instances_info, output_json_path):
    """
    将实例信息保存到JSON文件
    """
    with open(output_json_path, 'w') as f:
        json.dump(instances_info, f, indent=4)

def process_instance_masks(instance_mask, keypoints_list, output_json_path):
    """
    处理实例mask和关键点，生成最终的JSON结果
    """
    # 1. 为每个关键点分配实例ID
    instance_ids = find_instance_for_keypoints(instance_mask, keypoints_list)

    # 2. 按实例ID分组关键点
    instance_keypoints = {}
    for idx, instance_id in enumerate(instance_ids):
        if instance_id not in instance_keypoints:
            instance_keypoints[instance_id] = []
        instance_keypoints[instance_id].append(keypoints_list[idx])

    # 3. 对每个实例的关键点按最短路径排序
    instances_info = []
    for instance_id, keypoints in instance_keypoints.items():
        sorted_keypoints = tsp_3d(np.array(keypoints))
        total_length = calculate_total_length(sorted_keypoints)
        
        # 4. 将结果存入字典
        instance_info = {
            "id": int(instance_id),
            "points": [[float(z), float(y), float(x)] for z, y, x in sorted_keypoints],
            "length": float(total_length)
        }
        instances_info.append(instance_info)

    # 5. 保存结果到JSON文件
    save_results_to_json(instances_info, output_json_path)

In [17]:
output_json_path = '/media/liushuo/data1/data/actin trace/actintomo/20200820/tomo/pp1033/MT/pp1033_mt_point.json'
process_instance_masks(final_instance_mask, local_maxi, output_json_path)
